Gemini Content Moderation

Install Dependencies
pip install google-genai pydantic

In [2]:
!pip install google-genai pydantic

Defaulting to user installation because normal site-packages is not writeable
  Using cached google_genai-2.14.0-py3-none-any.whl.metadata (55 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached google_auth-2.56.2-py3-none-any.whl.metadata (6.0 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached websockets-16.1.1-cp313-cp313-win_amd64.whl.metadata (7.0 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached pydantic_core-2.46.4-cp313-cp313-win_amd64.whl.metadata (6.7 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached pyasn1-0.6.4-py3-none


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
from enum import Enum
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

In [9]:
# Ensure your API key is configured
os.environ["GEMINI_API_KEY"] = "AQ.Ab8RN6KkmCMBC4Vkegi0o3ZBt434m9Q8WmO3iJzIzKqTcGnxgg"

In [5]:
# 1. Define your moderation categories using an Enum
class HarmCategoryEnum(str, Enum):
    SAFE = "Safe"
    TOXICITY = "Toxicity or Hate Speech"
    HARASSMENT = "Harassment or Cyberbullying"
    SEXUAL = "Sexually Explicit Content"
    VIOLENCE = "Violence or Dangerous Acts"
    PII = "Personally Identifiable Information"

In [6]:
# 2. Define the exact JSON structure you want Gemini to output
class ModerationResult(BaseModel):
    flagged: bool = Field(
        description="True if the text violates community standards or falls into a harmful category."
    )
    primary_category: HarmCategoryEnum = Field(
        description="The primary harm category matched. Select 'Safe' if the content passes."
    )
    confidence_score: float = Field(
        description="Confidence score between 0.0 (low confidence) and 1.0 (absolute certainty)."
    )
    reasoning: str = Field(
        description="A brief, 1-sentence explanation of why the text was flagged or cleared."
    )

In [11]:
def moderate_content(user_text: str) -> ModerationResult:
    client = genai.Client()
    
    # Define system instructions to give Gemini its persona and rules
    system_instruction = (
        "You are an enterprise content moderation system. Analyze the user text objectively. "
        "Ignore spelling attempts to bypass filters (e.g., symbol substitution). Do not moralize, "
        "simply classify the text according to the provided schema instructions."
    )
    
    # CRITICAL STEP: Turn off internal filters so Gemini can safely ingest the bad text to evaluate it.
    disable_internal_safety = [
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
            threshold=types.HarmBlockThreshold.BLOCK_NONE,
        ),
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
            threshold=types.HarmBlockThreshold.BLOCK_NONE,
        ),
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
            threshold=types.HarmBlockThreshold.BLOCK_NONE,
        ),
        types.SafetySetting(
            category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
            threshold=types.HarmBlockThreshold.BLOCK_NONE,
        ),
    ]

    # Call Gemini with the structured configuration
    response = client.models.generate_content(
        model="gemini-3.5-flash", # Use Flash for ultra-fast, cheap classification
        contents=f"Please moderate the following text:\n\n{user_text}",
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            safety_settings=disable_internal_safety,
            temperature=0.0, # Forces deterministic, consistent classifications
            response_mime_type="application/json", # Tells Gemini to speak JSON
            response_schema=ModerationResult,     # Enforces the Pydantic structural format
        ),
    )
    
    # Automatically returns the output parsed directly into your Pydantic object
    return response.parsed

# --- TEST THE MODERATOR ---

In [8]:
test_prompts = [
    "Hey! I really love the project you built, do you have a github link?",
    "You are an idiot and nobody likes you, I hope you lose your job tomorrow.",
    "Can you help me write a Python script that scrapes headlines from a website?"
]

In [13]:
print("--- Running Moderation Tests ---")
for text in test_prompts:
    result = moderate_content(text)
    print(f"\n[Input]: \"{text}\"")
    print(f" Flagged: {result.flagged}")
    print(f" Category: {result.primary_category.value}")
    print(f" Confidence: {result.confidence_score}")
    print(f" Reason: {result.reasoning}")

--- Running Moderation Tests ---

[Input]: "Hey! I really love the project you built, do you have a github link?"
 Flagged: False
 Category: Safe
 Confidence: 1.0
 Reason: The text is a polite and safe inquiry about a project's GitHub repository.

[Input]: "You are an idiot and nobody likes you, I hope you lose your job tomorrow."
 Flagged: True
 Category: Harassment or Cyberbullying
 Confidence: 0.98
 Reason: The text contains direct personal insults and malicious wishes directed at an individual, which constitutes harassment.

[Input]: "Can you help me write a Python script that scrapes headlines from a website?"
 Flagged: False
 Category: Safe
 Confidence: 1.0
 Reason: The request is a benign programming query about web scraping and does not violate any safety policies.
